# Modelling


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from olist import config as cfg, io_utils as io, modeling as ml
from olist.features import PURCHASE_TIME_FEATURES

df = io.load_ml("order_model_base")
print("shared ML base table:", df.shape)
print("\nchronological split, identical for every project:")
summary = (
    df.groupby("split")
    .agg(orders=("Order_Id", "size"),
         first=("Order_Purchase_Timestamp", "min"), last=("Order_Purchase_Timestamp", "max"),
         late_rate=("late_delivery_flag", "mean"))
    .reindex(["train", "valid", "test"])
)
summary["late_rate"] = summary["late_rate"].round(4)
print(summary.to_string())

shared ML base table: (99441, 48)

chronological split, identical for every project:
       orders               first                last  late_rate
split                                                           
train   73577 2016-09-04 21:15:19 2018-04-30 23:47:26     0.0879
valid   13040 2018-05-01 00:02:11 2018-06-30 23:59:49     0.0497
test    12824 2018-07-01 00:25:07 2018-10-17 17:30:18     0.0748


In [2]:
# The shared feature contract, audited once here for the flagship target.
audit = ml.leakage_audit([c for c in PURCHASE_TIME_FEATURES if c in df.columns], "late_delivery_flag")
print(f"{len(audit)} features, all allowed.\n")
print("Banned by construction (config.POST_OUTCOME_COLUMNS):")
for c in cfg.POST_OUTCOME_COLUMNS:
    print(f"  - {c}")

39 features, all allowed.

Banned by construction (config.POST_OUTCOME_COLUMNS):
  - Order_Approved_At
  - Order_Delivered_Carrier_Date
  - Order_Delivered_Customer_Date
  - Order_Status
  - approval_hours
  - carrier_hours
  - last_mile_hours
  - total_delivery_hours
  - delivery_days
  - delivery_delay_days
  - late_delivery_flag
  - cancelled_flag
  - Is_Approved
  - Is_Shipped
  - Is_Delivered
  - review_score


## 1. Overview
